# Statistic

Input `model_name` and `seed_list`, then print per-partition statistics after excluding `cluster_id = -1`.


In [10]:
import csv
import math
from pathlib import Path


In [11]:
def read_csv_rows(path):
    path = Path(path)
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def load_full_summary_rows(out_root, model_name, space_name):
    summary_path = Path(out_root) / model_name / space_name / "summary.csv"
    return read_csv_rows(summary_path)

def load_random_summary_rows(out_root, model_name, space_name, seed):
    summary_path = (
        Path(out_root) / model_name / space_name / "random" / f"seed_{int(seed)}" / f"summary_random_pca_seed_{int(seed)}.csv"
    )
    return read_csv_rows(summary_path)

def mean_std(values):
    if not values:
        return 0.0, 0.0
    mean = sum(values) / len(values)
    var = sum((x - mean) ** 2 for x in values) / len(values)
    return mean, math.sqrt(var)

def compute_partition_stats(cluster_csv_path):
    rows = read_csv_rows(cluster_csv_path)
    valid_rows = [row for row in rows if int(row["cluster_id"]) != -1]
    cluster_sizes = {}
    probabilities = []
    for row in valid_rows:
        cluster_id = int(row["cluster_id"])
        cluster_sizes[cluster_id] = cluster_sizes.get(cluster_id, 0) + 1
        probabilities.append(float(row["probability"]))
    size_values = list(cluster_sizes.values())
    mean_cluster_size, std_cluster_size = mean_std(size_values)
    mean_probability, std_probability = mean_std(probabilities)
    return {
        "n_clusters": len(cluster_sizes),
        "mean_cluster_size": mean_cluster_size,
        "std_cluster_size": std_cluster_size,
        "mean_probability": mean_probability,
        "std_probability": std_probability,
    }

def print_partition_stats(partition_type, seed, pca_dim, stats):
    print(
        f"partition_type={partition_type} | seed={seed} | pca_dim={pca_dim} | "
        f"n_clusters={stats['n_clusters']} | "
        f"cluster_size_mean={stats['mean_cluster_size']:.6f} | cluster_size_std={stats['std_cluster_size']:.6f} | "
        f"probability_mean={stats['mean_probability']:.6f} | probability_std={stats['std_probability']:.6f}"
    )


In [12]:
out_root = "comp"
model_name = "mistralai/Mistral-7B-v0.1"
space_name = "output_proj"
seed_list = [0, 42, 1000, 9999]


In [13]:
for summary_row in load_full_summary_rows(out_root, model_name, space_name):
    stats = compute_partition_stats(summary_row["cluster_csv_path"])
    print_partition_stats("full", None, int(summary_row["pca_dim"]), stats)

for seed in seed_list:
    for summary_row in load_random_summary_rows(out_root, model_name, space_name, seed):
        stats = compute_partition_stats(summary_row["cluster_csv_path"])
        print_partition_stats("random", int(seed), int(summary_row["pca_dim"]), stats)


partition_type=full | seed=None | pca_dim=5 | n_clusters=308 | cluster_size_mean=12.094156 | cluster_size_std=23.411499 | probability_mean=0.943976 | probability_std=0.155331
partition_type=full | seed=None | pca_dim=142 | n_clusters=79 | cluster_size_mean=35.063291 | cluster_size_std=133.705641 | probability_mean=0.863848 | probability_std=0.320147
partition_type=full | seed=None | pca_dim=997 | n_clusters=647 | cluster_size_mean=11.340031 | cluster_size_std=24.034210 | probability_mean=0.927340 | probability_std=0.202781
partition_type=full | seed=None | pca_dim=2084 | n_clusters=883 | cluster_size_mean=11.283126 | cluster_size_std=19.271477 | probability_mean=0.942140 | probability_std=0.163894
partition_type=full | seed=None | pca_dim=3031 | n_clusters=939 | cluster_size_mean=10.905218 | cluster_size_std=12.723845 | probability_mean=0.947284 | probability_std=0.158139
partition_type=full | seed=None | pca_dim=3459 | n_clusters=953 | cluster_size_mean=10.657922 | cluster_size_std=6.

In [14]:
out_root = "comp"
model_name = "gpt-oss"
space_name = "output_proj"
seed_list = [0, 42, 1000, 9999, 1813382118, 827307999, 1627694678, 1911784257]


In [15]:
for summary_row in load_full_summary_rows(out_root, model_name, space_name):
    stats = compute_partition_stats(summary_row["cluster_csv_path"])
    print_partition_stats("full", None, int(summary_row["pca_dim"]), stats)

for seed in seed_list:
    for summary_row in load_random_summary_rows(out_root, model_name, space_name, seed):
        stats = compute_partition_stats(summary_row["cluster_csv_path"])
        print_partition_stats("random", int(seed), int(summary_row["pca_dim"]), stats)


partition_type=full | seed=None | pca_dim=6 | n_clusters=1203 | cluster_size_mean=42.197839 | cluster_size_std=509.467790 | probability_mean=0.978759 | probability_std=0.059391
partition_type=full | seed=None | pca_dim=182 | n_clusters=740 | cluster_size_mean=42.328378 | cluster_size_std=146.692449 | probability_mean=0.928540 | probability_std=0.096564
partition_type=full | seed=None | pca_dim=466 | n_clusters=1941 | cluster_size_mean=16.596084 | cluster_size_std=46.706982 | probability_mean=0.966580 | probability_std=0.057311
partition_type=full | seed=None | pca_dim=739 | n_clusters=2783 | cluster_size_mean=16.890047 | cluster_size_std=37.807712 | probability_mean=0.962381 | probability_std=0.053519
partition_type=full | seed=None | pca_dim=1591 | n_clusters=3625 | cluster_size_mean=18.097931 | cluster_size_std=33.780542 | probability_mean=0.951917 | probability_std=0.063139
partition_type=full | seed=None | pca_dim=2264 | n_clusters=3819 | cluster_size_mean=17.977743 | cluster_size_

In [16]:
def read_metric_rows(path):
    path = Path(path)
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def aggregate_scale_mean_std(rows, value_col):
    values = [float(row[value_col]) for row in rows]
    return mean_std(values)

def aggregate_random_cross_seed(rows, value_col, seed_list):
    seed_set = {int(seed) for seed in seed_list}
    per_seed = {}
    for row in rows:
        seed = int(row["pca_seed"])
        if seed in seed_set:
            per_seed.setdefault(seed, []).append(float(row[value_col]))
    seed_means = []
    for seed in seed_list:
        seed = int(seed)
        if seed not in per_seed:
            raise ValueError(f"Missing random rows for seed={seed}")
        seed_values = per_seed[seed]
        seed_means.append(sum(seed_values) / len(seed_values))
    return mean_std(seed_means)

def format_mean_std(mean_value, std_value):
    return f"{mean_value:.6f} ± {std_value:.6f}"

model_specs = [
    {
        "label": "Mistral",
        "model_name": "mistralai/Mistral-7B-v0.1",
        "perm_model_name": "permutation/mistralai/Mistral-7B-v0.1_rand",
    },
    {
        "label": "Mixtral",
        "model_name": "mistralai/Mixtral-8x7B-v0.1",
        "perm_model_name": "permutation/mistralai/Mixtral-8x7B-v0.1_rand",
    },
    {
        "label": "gpt-oss",
        "model_name": "gpt-oss",
        "perm_model_name": "permutation/gpt-oss_rand",
    },
]

metric_specs = [
    {
        "label": "morphology",
        "full_filename": "kondrak_global_summary.csv",
        "random_filename": "kondrak_global_summary_all_seeds.csv",
        "perm_filename": "kondrak_global_summary.csv",
        "value_col": "global_mean",
    },
    {
        "label": "multi-lingual",
        "full_filename": "script_entropy_summary.csv",
        "random_filename": "script_entropy_all_seeds.csv",
        "perm_filename": "script_entropy_summary.csv",
        "value_col": "mean_H_norm",
    },
]


In [17]:
seed_list = [0, 42, 1000, 9999, 1813382118, 827307999, 1627694678, 1911784257]

In [18]:
for model_spec in model_specs:
    for metric_spec in metric_specs:
        full_path = Path(out_root) / model_spec["model_name"] / space_name / metric_spec["full_filename"]
        random_path = Path(out_root) / model_spec["model_name"] / space_name / "random" / metric_spec["random_filename"]
        perm_path = Path(out_root) / model_spec["perm_model_name"] / space_name / metric_spec["perm_filename"]

        full_rows = read_metric_rows(full_path)
        random_rows = read_metric_rows(random_path)
        perm_rows = read_metric_rows(perm_path)

        full_mean, full_std = aggregate_scale_mean_std(full_rows, metric_spec["value_col"])
        random_mean, random_std = aggregate_random_cross_seed(random_rows, metric_spec["value_col"], seed_list)
        perm_mean, perm_std = aggregate_scale_mean_std(perm_rows, metric_spec["value_col"])

        print(
            f"{model_spec['label']} | {metric_spec['label']} | "
            f"Full={format_mean_std(full_mean, full_std)} | "
            f"Random={format_mean_std(random_mean, random_std)} | "
            f"Permutation={format_mean_std(perm_mean, perm_std)}"
        )


Mistral | morphology | Full=0.500235 ± 0.117081 | Random=0.501608 ± 0.001544 | Permutation=0.141519 ± 0.018264
Mistral | multi-lingual | Full=0.044238 ± 0.018380 | Random=0.044366 ± 0.000462 | Permutation=0.102490 ± 0.049296
Mixtral | morphology | Full=0.484474 ± 0.114237 | Random=0.486118 ± 0.001667 | Permutation=0.146256 ± 0.014447
Mixtral | multi-lingual | Full=0.046026 ± 0.010374 | Random=0.045823 ± 0.000502 | Permutation=0.095320 ± 0.030664
gpt-oss | morphology | Full=0.497264 ± 0.098663 | Random=0.499139 ± 0.001610 | Permutation=0.101210 ± 0.015891
gpt-oss | multi-lingual | Full=0.069646 ± 0.027079 | Random=0.068802 ± 0.000582 | Permutation=0.279915 ± 0.126088
